# Analysis II - Models and controlled experiments
**Evidence classes:** The first plots here are counterfactual calculations using the observed event timestamps from [Analysis I](01_observed_workloads.ipynb). They are not measured packets, response times, watts, or joules. A separate experimental results table is defined below; it remains empty until comparable service runs have been performed.

## Questions
1. Which batch size and maximum wait might reduce application sends while holding events briefly? Apply an explicitly defined count-or-timeout model to a fixed replay.
2. Which settings actually reduce bytes and joules per successful task without breaking delivery or latency targets? Answer with repeated, randomized-order measurements on the same testbed.
3. Which settings are nondominated? Compute a Pareto frontier only after correctness and service-level thresholds have been enforced.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display, Markdown
PATH = Path('../data/raw/intel/data.txt.gz')
assert PATH.exists(), 'Run Analysis I with the downloaded original Intel archive first.'
raw = pd.read_csv(PATH,sep=' ',names=['date','time','epoch','moteid','temperature','humidity','light','voltage'],header=None,compression='gzip',low_memory=False)
times = pd.to_datetime(raw.date.astype(str)+' '+raw.time.astype(str),errors='coerce')
ids = pd.to_numeric(raw.moteid,errors='coerce')
events = times[times.notna() & ids.between(1,54)].sort_values(kind='stable')
seconds = events.astype('int64').to_numpy()/1e9
print(f'Fixed replay: {len(seconds):,} timestamped, roster-valid events.')


## Explicit batch model
Within a single logical aggregation stream, flush when the batch reaches its record limit or at the first event's deadline, whichever comes first. An event arriving on or after the deadline begins the next batch. Flush the final partial batch at its deadline. Holding time is an **added waiting component**, not measured end-to-end delivery latency. The model assumes perfect delivery and ignores bytes, packetization, headers, retries, acknowledgements, CPU, and energy.


In [ ]:
def model_batches(times, max_count, max_wait_s):
    if max_count < 1 or max_wait_s < 0: raise ValueError('Invalid settings')
    if len(times) == 0: return 0,np.array([])
    waits = np.empty(len(times),dtype=float); start = 0; batches = 0
    for i in range(len(times)):
        if i > start and times[i] >= times[start]+max_wait_s:
            waits[start:i] = times[start]+max_wait_s-times[start:i]
            batches += 1; start = i
        if i-start+1 == max_count:
            waits[start:i+1] = times[i]-times[start:i+1]
            batches += 1; start = i+1
    if start < len(times):
        waits[start:] = times[start]+max_wait_s-times[start:]
        batches += 1
    return batches,waits

# Cases chosen so flush boundaries and waiting times are hand-checkable.
assert model_batches(np.array([0.,2.,10.]),8,5)[0] == 2
np.testing.assert_allclose(model_batches(np.array([0.,2.,10.]),8,5)[1],[5.,3.,5.])
np.testing.assert_allclose(model_batches(np.array([0.,2.,10.]),1,60)[1],[0.,0.,0.])
print('Two boundary checks passed.')


In [ ]:
rows=[]
for count in (1,8,32):
    for wait in (1,10,60):
        batches,holds = model_batches(seconds,count,wait)
        rows.append({'max_count':count,'max_wait_s':wait,'modelled_batches':batches,'modelled_reduction_pct':100*(1-batches/len(seconds)),'modelled_p95_hold_s':float(np.quantile(holds,.95))})
scenarios=pd.DataFrame(rows)
display(scenarios)
fig=px.line(scenarios,x='modelled_p95_hold_s',y='modelled_batches',color='max_count',symbol='max_wait_s',markers=True,hover_data=['modelled_reduction_pct'],title='6. Modelled sends versus modelled holding time; no energy measurement'); fig.show()
fig=px.bar(scenarios,x='max_count',y='modelled_reduction_pct',color='max_wait_s',barmode='group',title='7. Modelled reduction relative to one application send per event'); fig.show()
display(Markdown('**Technical:** These are outputs of a deterministic batching rule conditional on logged timestamps. They are not an estimate of packet, byte, CPU or energy savings; tightly clustered timestamps can make modelled holding time appear very small.  \n**Plain language:** Combining readings may mean fewer sends, but we still need to run the actual server to learn whether it saves electricity and meets delivery deadlines.'))


## Experimental protocol and result schema
Freeze topology, payload, hardware, software versions and workload replay. Alternate or randomize baseline and candidate order, repeat each setting, and measure idle separately. Use actual producer-to-queue-to-worker-to-database completion IDs. Measure wall watts over the task interval, per-edge bytes, p50/p95/p99 end-to-end latency, retries, losses and duplicates. Define acceptance thresholds before inspecting outcomes. Report uncertainty across runs and separate conclusions by workload and machine.

The following schema accepts measured results later. It intentionally contains **no fabricated observations**. A row is one repeated run, with `energy_j` measured from integrated wall watts and `successful_tasks` counted from verified completed IDs. Record environmental and instrumentation metadata in a companion protocol file.


In [ ]:
required=['configuration','workload','machine','run_id','energy_j','successful_tasks','bytes_total','p95_latency_ms','lost','duplicates']
results_path=Path('results/experimental_runs.csv')
if results_path.exists():
    runs=pd.read_csv(results_path)
    assert set(required).issubset(runs.columns), 'Missing required experimental columns'
    assert (runs.successful_tasks>0).all(), 'Cannot divide by zero completed tasks'
    runs['joules_per_success']=runs.energy_j/runs.successful_tasks
    display(runs.head())
else:
    runs=pd.DataFrame(columns=required)
    display(Markdown('**Measured experiment status:** no experimental run file exists yet. Pareto and energy claims remain pending.'))


## Pareto frontier for measured, valid runs
Enter the predeclared latency budget below once the task has a service-level requirement. Use configuration-level summaries and show individual runs to reveal variability. A configuration is eligible only if its repeated runs satisfy the correctness constraints. This exploratory frontier uses mean joules per successful task and mean p95 latency; final claims also need uncertainty intervals and a sensitivity analysis.


In [ ]:
LATENCY_BUDGET_MS=None  # Set from the task requirement before analysing measured runs.
if not runs.empty and LATENCY_BUDGET_MS is not None:
    valid=runs[(runs.lost==0)&(runs.duplicates==0)&(runs.p95_latency_ms<=LATENCY_BUDGET_MS)]
    grouped=valid.groupby(['configuration','workload','machine'],as_index=False).agg(joules_per_success=('joules_per_success','mean'),p95_latency_ms=('p95_latency_ms','mean'),bytes_total=('bytes_total','mean'),repeats=('run_id','nunique'))
    grouped['pareto']=False
    for (_, _),indices in grouped.groupby(['workload','machine']).groups.items():
        for i in indices:
            q=grouped.loc[list(indices)]
            dominated=((q.joules_per_success<=grouped.at[i,'joules_per_success'])&(q.p95_latency_ms<=grouped.at[i,'p95_latency_ms'])&((q.joules_per_success<grouped.at[i,'joules_per_success'])|(q.p95_latency_ms<grouped.at[i,'p95_latency_ms']))).any()
            grouped.at[i,'pareto']=not dominated
    display(grouped)
    fig=px.scatter(grouped,x='p95_latency_ms',y='joules_per_success',color='pareto',facet_col='workload',symbol='machine',hover_data=['configuration','repeats','bytes_total'],title='Measured Pareto candidates by workload and hardware'); fig.show()
else:
    display(Markdown('No measured Pareto frontier: provide repeated experimental runs and a predeclared latency budget.'))


## Interpretation boundary
A smaller count of application sends is a hypothesis about work avoided. A lower measured energy per *successful* task, with comparable reliability and latency, supports a testbed-specific sustainability claim. It does not automatically apply to other devices, workloads or facilities. See the [research plan](research-plan.md) and [assessment](assessment.md) for the next data collection steps.
